In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain.chat_models import init_chat_model

#llm = init_chat_model("anthropic:claude-opus-4-1-20250805", temperature=0)
llm = init_chat_model("gemini-2.5-flash",model_provider="google_genai", temperature=0)
response = llm.invoke("Hola, como estas?")
response.text

'Hola, estoy muy bien, gracias. ¿Y tú, cómo estás?'

In [3]:
response = llm.invoke("Que clima hace en la ciudad de Bogota?")
response.text

'Como modelo de lenguaje, no tengo acceso a información en tiempo real, por lo que no puedo decirte el clima exacto en este momento.\n\nPara obtener la información más precisa y actualizada, te recomiendo consultar:\n\n*   **Aplicaciones del clima** (como AccuWeather, The Weather Channel, Google Weather).\n*   **Sitios web de meteorología** (como el IDEAM en Colombia).\n*   **Noticieros locales.**\n\n**Sin embargo, te puedo dar una descripción general del clima típico de Bogotá:**\n\nBogotá tiene un **clima frío de alta montaña** debido a su elevación (aproximadamente 2640 metros sobre el nivel del mar).\n\n*   **Temperaturas:** Son frescas y relativamente constantes durante todo el año. Las temperaturas diurnas suelen oscilar entre los 18°C y 22°C, mientras que las nocturnas bajan a entre 7°C y 10°C. Rara vez hace mucho calor o mucho frío extremo.\n*   **Lluvias:** Es una ciudad con lluvias frecuentes, especialmente durante dos temporadas principales: de marzo a mayo y de septiembre a

In [4]:
system_prompt = """
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan.

Y tus productos son:
- Computadora
- Mouse
- Teclado
- Audifonos
- Mousepad
"""
messages = [
    ("system", system_prompt),
    ("user", "Dime los productos que ofreces en la tienda")
]
response = llm.invoke(messages)
response.text

'¡Claro! En nuestra tienda, ofrecemos los siguientes productos:\n\n*   **Computadoras**\n*   **Mouses**\n*   **Teclados**\n*   **Audífonos**\n*   **Mousepads**\n\n¿Hay algo en particular que te interese o en lo que te pueda ayudar?'

In [7]:
from langchain_core.tools import tool
import requests

@tool("get_products", description="Get the products that the store offers filter by price")
def get_products():
    # Connnect with API
    """Get the products that the store offers"""
    response = requests.get("https://api.escuelajs.co/api/v1/products")
    products = response.json()
    return "".join([f"{product['title']} - {product['price']}" for product in products])

In [6]:
get_products.invoke({})

'Classic Comfort Fit Joggers - 25Classic Comfort Drawstring Joggers - 79Classic Red Jogger Sweatpants - 98Classic Navy Blue Baseball Cap - 61Classic Red Pullover Hoodie - 10Classic Red Baseball Cap - 35Classic Blue Baseball Cap - 86Classic Black Baseball Cap - 58Classic Olive Chino Shorts - 84Classic White Tee - Timeless Style and Comfort - 73Sleek White & Orange Wireless Gaming Controller - 69Classic Black T-Shirt - 35Sleek Wireless Headphone & Inked Earbud Set - 44Classic White Crew Neck T-Shirt - 39Sleek Comfort-Fit Over-Ear Headphones - 28Classic High-Waisted Athletic Shorts - 43Efficient 2-Slice Toaster - 48Sleek Wireless Computer Mouse - 10Stylish Red & Silver Over-Ear Headphones - 39Sleek Modern Laptop with Ambient Lighting - 43Sleek Modern Laptop for Professionals - 97Sleek Smartwatch with Vibrant Display - 16Sleek Mirror Finish Phone Case - 27Sleek Modern Leather Sofa - 53Mid-Century Modern Wooden Dining Table - 24Elegant Golden-Base Stone Top Dining Table - 66Modern Elegance 

In [2]:
from langchain_core.tools import tool
import requests
@tool("get_weather", description="Get the weather of a city")
def get_weather(city: str):
    response = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1")
    data = response.json()
    latitude = data["results"][0]["latitude"]
    longitude = data["results"][0]["longitude"]
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true")
    data = response.json()
    response = f"The weather in {city} is {data["current_weather"]["temperature"]}C with {data["current_weather"]["windspeed"]}km/h of wind."
    return response

get_weather.invoke({"city": "bogota"})

'The weather in bogota is 19.9C with 14.8km/h of wind.'

In [12]:
system_prompt = """
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan y dar el clima de la ciudad

Tus tools son:
- get_products: para obtener los productos que ofreces en la tienda
- get_weather: para obtener el clima de la ciudad
"""
messages = [
    ("system", system_prompt),
    ("user", "Dime los productos que ofreces en la tienda")
]
llm_with_tools = llm.bind_tools([get_products, get_weather])
response = llm_with_tools.invoke(messages)
response.pretty_print()
response.tool_calls

================================== Ai Message ==================================
Tool Calls:
  get_products (19df1ac1-50ee-480c-a1c7-d164dac0c6de)
 Call ID: 19df1ac1-50ee-480c-a1c7-d164dac0c6de
  Args:


[{'name': 'get_products',
  'args': {},
  'id': '19df1ac1-50ee-480c-a1c7-d164dac0c6de',
  'type': 'tool_call'}]

In [13]:
messages = [
    ("system", system_prompt),
    ("user", "Hola, que tal?")
]
response = llm_with_tools.invoke(messages)
response.text

'¡Hola! Estoy aquí para ayudarte a encontrar lo que necesitas. ¿Estás buscando algún producto en particular o te gustaría saber qué tenemos en la tienda? También puedo darte el clima de alguna ciudad si lo deseas.'

In [15]:
system_prompt = """
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan y dar el clima de la ciudad

Tus tools son:
- get_products: para obtener los productos que ofreces en la tienda
- get_weather: para obtener el clima de la ciudad
"""
messages = [
    ("system", system_prompt),
    ("user", "Cual es el clima en la capital de Colombia?")
]
llm_with_tools = llm.bind_tools([get_products, get_weather])
response = llm_with_tools.invoke(messages)
response.tool_calls

[{'name': 'get_weather',
  'args': {'city': 'Bogotá'},
  'id': '07dbc02a-b74f-4e52-9c97-1b251bc9912d',
  'type': 'tool_call'}]